In [1]:
import pandas as pd
from scipy import stats
from sqlalchemy import create_engine

username = "root"
password = "2003"
host = "localhost"
port = "3306"
database = "cart2insights"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


In [2]:
ttest_df = pd.read_sql("""
    SELECT o.order_id, o.order_delivered_customer_date, o.order_estimated_delivery_date, r.review_score
    FROM orders o
    JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
""", con=engine)

ttest_df["order_delivered_customer_date"] = pd.to_datetime(ttest_df["order_delivered_customer_date"])
ttest_df["order_estimated_delivery_date"] = pd.to_datetime(ttest_df["order_estimated_delivery_date"])
ttest_df["is_delayed"] = ttest_df["order_delivered_customer_date"] > ttest_df["order_estimated_delivery_date"]

delayed_scores = ttest_df[ttest_df["is_delayed"] == True]["review_score"]
ontime_scores = ttest_df[ttest_df["is_delayed"] == False]["review_score"]

print(f"Delayed orders: n={len(delayed_scores)}, mean review = {delayed_scores.mean():.2f}")
print(f"On-time orders: n={len(ontime_scores)}, mean review = {ontime_scores.mean():.2f}")

t_stat, p_value = stats.ttest_ind(delayed_scores, ontime_scores, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject null hypothesis - delayed orders DO have significantly different review scores.")
else:
    print("\nResult: Fail to reject null hypothesis - no significant difference.")

Delayed orders: n=7701, mean review = 2.57
On-time orders: n=88658, mean review = 4.29

T-statistic: -89.5508
P-value: 0.0000000000

Result: Reject null hypothesis - delayed orders DO have significantly different review scores.


In [3]:
anova_df = pd.read_sql("""
    SELECT p.product_category_name, oi.price
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.product_category_name IS NOT NULL AND p.product_category_name != 'unknown'
""", con=engine)

# Only keep categories with a reasonable sample size for a fair test
category_counts = anova_df["product_category_name"].value_counts()
valid_categories = category_counts[category_counts >= 30].index
anova_df = anova_df[anova_df["product_category_name"].isin(valid_categories)]

groups = [group["price"].values for name, group in anova_df.groupby("product_category_name")]

f_stat, p_value = stats.f_oneway(*groups)
print(f"Number of categories tested: {len(groups)}")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject null hypothesis - average order value DOES differ significantly across categories.")
else:
    print("\nResult: Fail to reject null hypothesis - no significant difference.")

# Show top 10 categories by average price for context
top_categories = anova_df.groupby("product_category_name")["price"].mean().sort_values(ascending=False).head(10)
print("\nTop 10 categories by average price:")
print(top_categories)

Number of categories tested: 66
F-statistic: 212.4524
P-value: 0.0000000000

Result: Reject null hypothesis - average order value DOES differ significantly across categories.

Top 10 categories by average price:
product_category_name
pcs                                 1098.340542
portateis_casa_forno_e_cafe          624.285658
eletrodomesticos_2                   476.124958
agro_industria_e_comercio            342.124858
instrumentos_musicais                281.616000
eletroportateis                      280.778468
telefonia_fixa                       225.693182
construcao_ferramentas_seguranca     208.992371
relogios_presentes                   201.135984
climatizacao                         185.269226
Name: price, dtype: float64


In [4]:
chi_df = pd.read_sql("""
    SELECT p.payment_type, o.order_status
    FROM order_payments p
    JOIN orders o ON p.order_id = o.order_id
""", con=engine)

contingency_table = pd.crosstab(chi_df["payment_type"], chi_df["order_status"])
print("Contingency Table:")
print(contingency_table)

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nChi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject null hypothesis - payment method IS significantly associated with order status.")
else:
    print("\nResult: Fail to reject null hypothesis - no significant association.")
    

Contingency Table:
order_status  approved  canceled  created  delivered  invoiced  processing  \
payment_type                                                                 
boleto               0        95        2      19191        67          70   
credit_card          2       444        3      74586       239         224   
debit_card           0         7        0       1486         6           2   
voucher              0       115        0       5493        13          23   

order_status  shipped  unavailable  
payment_type                        
boleto            209          150  
credit_card       851          446  
debit_card         22            6  
voucher            84           47  

Chi-square statistic: 211.5082
Degrees of freedom: 21
P-value: 0.0000000000

Result: Reject null hypothesis - payment method IS significantly associated with order status.
